# 01 — Tuples, Immutability Guarantees, and Spatial Coordinates

## Theoretical Foundations of Tuples

In discrete mathematics and computing, a **tuple** represents a finite, ordered sequence of elements. While a dynamic array or list represents a homogeneous, expandable collection of indeterminate length, a tuple mathematically corresponds to a single point or element within a Cartesian product:

$$(x_1, x_2, \dots, x_k) \in S_1 	imes S_2 	imes \dots 	imes S_k$$

In Python, tuples provide two critical software engineering properties:
1. **Positional Invariance & Immutability:** Once instantiated, the size and object references of a tuple cannot be reassigned, appended, or popped. This ensures thread safety, integrity guarantees for mission-critical constants, and hashability when all contained elements are themselves immutable.
2. **Heterogeneous Record Semantics:** Whereas lists are typically intended for homogeneous streams of items ($[x_1, x_2, \dots]$), tuples model heterogeneous structured records (e.g., an RGB color triplet `(255, 128, 0)` or a bounding box `(x_min, y_min, x_max, y_max)`).

In this notebook, we explore tuple construction, immutability contracts, memory efficiencies over lists, unpacking protocols, and apply tuples to solve quadratic equations and evaluate bounding box Intersection over Union (IoU).


In [ ]:
# 1. Tuple Construction Syntax
# Parentheses syntax
coordinate_2d = (10, 20)

# Comma-separated syntax without enclosing parentheses
coordinate_3d = 10, 20, 30

# Singleton tuple (requires a trailing comma to distinguish from grouping parentheses)
singleton_tuple = (42,)
not_a_tuple = 42

print(f"2D coordinate: {coordinate_2d}, type: {type(coordinate_2d).__name__}")
print(f"3D coordinate: {coordinate_3d}, type: {type(coordinate_3d).__name__}")
print(f"Singleton tuple: {singleton_tuple}, type: {type(singleton_tuple).__name__}")
print(f"Parenthesized int: {not_a_tuple}, type: {type(not_a_tuple).__name__}")

assert isinstance(singleton_tuple, tuple)
assert isinstance(not_a_tuple, int)

## Sequence Operators and Immutability Verification

Tuples conform to Python's sequence protocol:
- Concatenation: `t1 + t2` produces a new tuple combining both sequences.
- Repetition: `t * k` produces a new tuple repeating elements $k$ times.
- Slicing: `t[start:stop:step]` returns a new sub-tuple.
- Element assignment (`t[i] = x`) is explicitly forbidden and raises a `TypeError` at runtime.


In [ ]:
# 2. Operators and Immutability
tuple_a = (1, 2)
tuple_b = (3, 4)

# Concatenation and repetition
combined = tuple_a + tuple_b
repeated = tuple_a * 3

print(f"Concatenation: {combined}")
print(f"Repetition: {repeated}")

# Attempting mutation raises TypeError
try:
    combined[0] = 999  # type: ignore[index]
except TypeError as error:
    print(f"Caught expected immutability error: {error}")

assert combined == (1, 2, 3, 4)
assert repeated == (1, 2, 1, 2, 1, 2)

## Shallow Immutability and Reference Semantics

A critical nuance in CPython's object model is **shallow immutability**. A tuple stores a fixed sequence of object pointers (`PyObject*`). While the pointers themselves cannot be replaced or reordered, if a pointer references a **mutable** object (such as a `list` or `dict`), the internal state of that referenced object can still be mutated in place.


In [ ]:
# 3. Shallow Immutability
# A tuple containing an integer, a string, and a mutable list
record = (1, "alpha", [10, 20])
print(f"Before mutation: {record}")

# Mutating the internal list via its reference
record[2].append(30)
record[2][0] = 99
print(f"After mutation:  {record}")

assert record == (1, "alpha", [99, 20, 30])

## Memory Footprint: Tuples vs. Dynamic Arrays (Lists)

Because tuples have fixed lengths known at instantiation time, CPython optimizes `PyTupleObject` memory allocation:
1. Tuples do not allocate extra buffer capacity for future append operations (zero over-allocation).
2. Small tuples (up to 20 elements) are cached internally by CPython upon deallocation to accelerate future instantiation without calling system `malloc()`.

Consequently, tuples consume strictly less memory than equivalent lists.


In [ ]:
# 4. Memory Footprint Comparison
import sys

elements = list(range(100))
list_obj = list(elements)
tuple_obj = tuple(elements)

list_bytes = sys.getsizeof(list_obj)
tuple_bytes = sys.getsizeof(tuple_obj)

print(f"Size of 100-element list:  {list_bytes} bytes")
print(f"Size of 100-element tuple: {tuple_bytes} bytes")
print(
    f"Memory saved by tuple:    {list_bytes - tuple_bytes} bytes ({(1 - tuple_bytes / list_bytes) * 100:.1f}%)"
)

assert tuple_bytes < list_bytes

## Tuple Packing, Unpacking, and Swapping

Python enables structural pattern assignment through **tuple packing and unpacking**:
- **Packing:** Grouping values into a tuple without explicit parenthesis.
- **Unpacking:** Binding tuple elements directly into separate variable identifiers.
- **Starred Unpacking (`*rest`):** Capturing indeterminate intermediate sequences into a list.
- **Atomic Variable Swap:** `a, b = b, a` evaluates the right-hand tuple first on the evaluation stack before binding the left-hand identifiers, avoiding temporary storage variables.


In [ ]:
# 5. Unpacking Protocols and Variable Swapping
# Basic unpacking
point = (1920, 1080)
width, height = point
assert width == 1920 and height == 1080

# Starred unpacking
detection_record = ("car", 0.94, 120, 85, 340, 290)
label, confidence, *bbox = detection_record
print(f"Label: {label}, Confidence: {confidence}, Bounding Box: {bbox}")
assert label == "car" and confidence == 0.94 and bbox == [120, 85, 340, 290]

# Variable swapping
val_x = 42
val_y = 99
val_x, val_y = val_y, val_x
assert val_x == 99 and val_y == 42
print(f"Swapped: val_x={val_x}, val_y={val_y}")

## Application: Solving Quadratic Equations with Immutable Return Types

Consider the general quadratic equation:

$$ax^2 + bx + c = 0 \quad (a \neq 0)$$

The discriminant is given by:

$$\Delta = b^2 - 4ac$$

- If $\Delta > 0$, there exist two distinct real roots: $x_{1,2} = \frac{-b \pm \sqrt{\Delta}}{2a}$.
- If $\Delta = 0$, there exists a single repeated real root: $x = \frac{-b}{2a}$.
- If $\Delta < 0$, there are no real roots (empty solution set).

Returning roots as a tuple guarantees that downstream consumers cannot accidentally tamper with the returned mathematical solutions.


In [ ]:
# 6. Quadratic Equation Solver
import math


def solve_quadratic(a: float, b: float, c: float) -> tuple[float, ...]:
    """Solves ax^2 + bx + c = 0 and returns real roots as an immutable tuple."""
    if a == 0:
        if b == 0:
            return ()
        return (-c / b,)

    delta = (b**2) - (4 * a * c)
    if delta > 0:
        sqrt_delta = math.sqrt(delta)
        r1 = (-b + sqrt_delta) / (2 * a)
        r2 = (-b - sqrt_delta) / (2 * a)
        return tuple(sorted((r1, r2)))
    if delta == 0:
        return (-b / (2 * a),)
    return ()


# Test cases
two_roots = solve_quadratic(1, -5, 6)
one_root = solve_quadratic(1, -2, 1)
no_roots = solve_quadratic(1, 0, 1)

print(f"x^2 - 5x + 6 = 0 roots: {two_roots}")
print(f"x^2 - 2x + 1 = 0 roots: {one_root}")
print(f"x^2 + 1 = 0 roots:      {no_roots}")

assert two_roots == (2.0, 3.0)
assert one_root == (1.0,)
assert no_roots == ()

## Spatial Coordinate Modeling: Intersection over Union (IoU)

In computer vision object detection pipelines (e.g., YOLO, Faster R-CNN), bounding boxes are parameterized as 4-element coordinate tuples:

$$B = (x_{\min}, y_{\min}, x_{\max}, y_{\max})$$

Given two candidate boxes $A$ and $B$, the **Intersection over Union (IoU)** evaluates spatial overlap:

$$\text{IoU}(A, B) = \frac{\text{Area}(A \cap B)}{\text{Area}(A \cup B)} = \frac{\text{Area}(A \cap B)}{\text{Area}(A) + \text{Area}(B) - \text{Area}(A \cap B)}$$

The coordinates of the intersection rectangle $(x_A, y_A, x_B, y_B)$ are determined by:

$$x_A = \max(x_{\min}^A, x_{\min}^B), \quad y_A = \max(y_{\min}^A, y_{\min}^B)$$
$$x_B = \min(x_{\max}^A, x_{\max}^B), \quad y_B = \min(y_{\max}^A, y_{\max}^B)$$
$$\text{Width}_{\cap} = \max(0, x_B - x_A), \quad \text{Height}_{\cap} = \max(0, y_B - y_A)$$


In [ ]:
# 7. Intersection over Union Implementation
def compute_iou(
    box_a: tuple[float, float, float, float],
    box_b: tuple[float, float, float, float],
) -> float:
    """Computes IoU between two axis-aligned bounding boxes (xmin, ymin, xmax, ymax)."""
    x_a = max(box_a[0], box_b[0])
    y_a = max(box_a[1], box_b[1])
    x_b = min(box_a[2], box_b[2])
    y_b = min(box_a[3], box_b[3])

    inter_w = max(0.0, x_b - x_a)
    inter_h = max(0.0, y_b - y_a)
    inter_area = inter_w * inter_h

    area_a = max(0.0, box_a[2] - box_a[0]) * max(0.0, box_a[3] - box_a[1])
    area_b = max(0.0, box_b[2] - box_b[0]) * max(0.0, box_b[3] - box_b[1])

    union_area = area_a + area_b - inter_area
    if union_area <= 0.0:
        return 0.0

    return inter_area / union_area


# Verification with geometric boxes
box_gt = (0.0, 0.0, 5.0, 5.0)  # Area = 25
box_pred = (2.5, 2.5, 7.5, 7.5)  # Area = 25
# Intersection: [2.5, 5.0] x [2.5, 5.0] -> Area = 2.5 * 2.5 = 6.25
# Union: 25 + 25 - 6.25 = 43.75
# Expected IoU: 6.25 / 43.75 = 1 / 7 ~= 0.142857

computed_iou = compute_iou(box_gt, box_pred)
print(f"Box GT:   {box_gt}")
print(f"Box Pred: {box_pred}")
print(f"Computed IoU: {computed_iou:.6f}")

assert abs(computed_iou - (1.0 / 7.0)) < 1e-5
assert compute_iou(box_gt, box_gt) == 1.0  # Perfect overlap
assert compute_iou(box_gt, (10.0, 10.0, 15.0, 15.0)) == 0.0  # Disjoint